## Creating table with CSV file

##### Use google drive storage

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##### Load the CSV file

In [16]:
import pandas as pd
file_path = "/content/drive/MyDrive/AD-Tech.csv"
df = pd.read_csv(file_path)

##### Creating connection with SQL Lite

In [17]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("smartgoaldb.db")
conn.execute("DROP TABLE IF EXISTS fact_ad_performance")

### Creating the fact_ad_performance table

* Schema validation
* Consistency

In [18]:
conn.execute("""
CREATE TABLE fact_ad_performance (
    date TEXT,
    site_id INTEGER,
    ad_type_id INTEGER,
    geo_id INTEGER,
    device_category_id INTEGER,
    advertiser_id INTEGER,
    order_id INTEGER,
    line_item_type_id INTEGER,
    os_id INTEGER,
    integration_type_id INTEGER,
    monetization_channel_id INTEGER,
    ad_unit_id INTEGER,
    total_impressions INTEGER,
    total_revenue REAL,
    viewable_impressions INTEGER,
    measurable_impressions INTEGER,
    revenue_share_percent REAL
)
""")

### Store data into DB

In [19]:
write_method = "append"
df.to_sql("fact_ad_performance", conn, if_exists=write_method, index=False)

567291

##### Make index

In [20]:
conn.execute("CREATE INDEX idx_date_adv ON fact_ad_performance(date, advertiser_id)")
conn.execute("CREATE INDEX idx_geo_device ON fact_ad_performance(geo_id, device_category_id)")
conn.commit()

### Creating queries

* revenue and total impressions per advertiser_id

In [23]:
query = """
SELECT
    advertiser_id,
    SUM(total_revenue) AS revenue,
    SUM(total_impressions) AS impressions
FROM fact_ad_performance
GROUP BY advertiser_id
ORDER BY revenue DESC
"""

pd.read_sql(query, conn)

,advertiser_id,revenue,impressions
0,79,29041.7864,13355234
1,16,4611.8754,1014464
2,88,1306.4286,871472
3,8,1204.0162,531210
4,2634,869.5090,514380
5,97,766.9524,591886
6,90,760.7848,271314
7,96,561.1734,299830
8,2644,231.0180,154012
9,139,209.3230,63340


### Getting eCPM

In [24]:
query = """
SELECT
    advertiser_id,
    (SUM(total_revenue) / SUM(total_impressions)) * 1000 AS ecpm
FROM fact_ad_performance
GROUP BY advertiser_id
"""

pd.read_sql(query, conn)

,advertiser_id,ecpm
0,8,2.266554
1,16,4.546120
2,79,2.174562
3,84,0.002931
4,88,1.499106
5,90,2.804075
6,96,1.871639
7,97,1.295777
8,139,3.304752
9,2089,0.000000


### Getting viability rate

In [25]:
query = """
SELECT
    SUM(viewable_impressions) * 1.0 /
    SUM(measurable_impressions) AS viewability_rate
FROM fact_ad_performance
"""

pd.read_sql(query, conn)

,viewability_rate
0,0.399289


### Revenue per device category id

In [26]:
query = """
SELECT
    geo_id,
    device_category_id,
    SUM(total_revenue) AS revenue
FROM fact_ad_performance
GROUP BY geo_id, device_category_id
ORDER BY revenue DESC
"""

pd.read_sql(query, conn)

,geo_id,device_category_id,revenue
0,187,1,15910.0872
1,187,2,14555.0220
2,187,3,3766.5416
3,33,1,1226.3840
4,33,2,885.1750
...,...,...,...
603,212,3,0.0000
604,221,2,0.0000
605,222,1,0.0000
606,225,2,0.0000


## Using multi dimensions

In [27]:
conn.execute("""
CREATE TABLE IF NOT EXISTS dim_date (
    date TEXT PRIMARY KEY
)
""")

In [28]:
conn.execute("""
CREATE TABLE IF NOT EXISTS dim_advertiser (
    advertiser_id INTEGER PRIMARY KEY
)
""")

In [29]:
conn.execute("""
CREATE TABLE IF NOT EXISTS dim_geo (
    geo_id INTEGER PRIMARY KEY
)
""")

In [30]:
conn.execute("""
CREATE TABLE IF NOT EXISTS dim_device (
    device_category_id INTEGER PRIMARY KEY
)
""")

### Populate dimensions with DF

In [31]:
df[['date']].drop_duplicates().to_sql("dim_date", conn, if_exists=write_method, index=False)

df[['advertiser_id']].drop_duplicates().to_sql("dim_advertiser", conn, if_exists=write_method, index=False)

df[['geo_id']].drop_duplicates().to_sql("dim_geo", conn, if_exists=write_method, index=False)

df[['device_category_id']].drop_duplicates().to_sql("dim_device", conn, if_exists=write_method, index=False)

5

### Creating new fact table

In [32]:
conn.execute("DROP TABLE IF EXISTS fact_ad_performance")

conn.execute("""
CREATE TABLE fact_ad_performance (
    date TEXT,
    advertiser_id INTEGER,
    geo_id INTEGER,
    device_category_id INTEGER,
    total_impressions INTEGER,
    total_revenue REAL,
    viewable_impressions INTEGER,
    measurable_impressions INTEGER,
    revenue_share_percent REAL,

    FOREIGN KEY (date) REFERENCES dim_date(date),
    FOREIGN KEY (advertiser_id) REFERENCES dim_advertiser(advertiser_id),
    FOREIGN KEY (geo_id) REFERENCES dim_geo(geo_id),
    FOREIGN KEY (device_category_id) REFERENCES dim_device(device_category_id)
)
""")

In [33]:
df[['date','advertiser_id','geo_id','device_category_id',
    'total_impressions','total_revenue',
    'viewable_impressions','measurable_impressions',
    'revenue_share_percent'
]].to_sql("fact_ad_performance", conn, if_exists=write_method, index=False)

567291

### Getting simple aggregation consult

In [34]:
query = """
SELECT
  d.advertiser_id,
  SUM(f.total_revenue)
FROM fact_ad_performance f
JOIN dim_advertiser d ON f.advertiser_id = d.advertiser_id
GROUP BY d.advertiser_id;
"""

pd.read_sql(query, conn)

,advertiser_id,SUM(f.total_revenue)
0,8,1204.0162
1,16,4611.8754
2,79,29041.7864
3,84,0.2520
4,88,1306.4286
5,90,760.7848
6,96,561.1734
7,97,766.9524
8,139,209.3230
9,2089,0.0000


### Showing star schema

In [35]:
!pip install eralchemy

In [36]:
from eralchemy import render_er

render_er("sqlite:///smartgoaldb.db", "er_diagram.png")